In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import torch
from scripts.utils.load_model import load_model, ModelName
from scripts.utils.load_dataset import load_dataset


model, dataset_transform = load_model(ModelName.RESNET50, dataset_name="imagenet", device="cuda:0")
dl_train, dl_eval, dl_test = load_dataset("imagenet", transform=dataset_transform, train_batch=128, eval_batch=128)

In [ ]:
model

In [ ]:
from ulib import StopCriteria, PertModule, ActivationExtractor
from ulib.attacks.fg_uap import FG_UAP
import torch

import time
from ulib.utils.trackers import MetricTracker

stop = StopCriteria(max_epochs=20, max_time=20 * 60)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=10 / 255, random_init=True)
optimizer = torch.optim.Adam(pert_model.parameters(), lr=0.02)
extractor = ActivationExtractor(model, "model.fc", exact_match=True, capture_output=False)


TARGET_CLASSES = [
    24,
    # 99,
    245,
    # 344,
    471,
    # 555,
    661,
    # 701,
    802,
    # 919,
]


for i, tgt in enumerate(TARGET_CLASSES):
    dl_train.tensors[1].fill_(tgt)  # set all labels to the target class
    dl_eval.tensors[1].fill_(tgt)
    dl_test.tensors[1].fill_(tgt)

    metric_tracker = MetricTracker.create(
        time.strftime("%Y-%m-%d_%H-%M-%S"),
        f"target_class_{tgt}",
        kind="wandb",
        project="CLF-UPD",
        root_dir="logs",
    )

    attack = FG_UAP(
        pert_model=pert_model,
        optimizer=optimizer,
        activ_extractor=extractor,
        # scheduler=scheduler,
        mixed_precision=False,
        targeted=True,
        metric_tracker=metric_tracker,
    )

    pert = attack.fit(dl_train, dl_eval, stop)

In [ ]:
from ulib.evaluator import ExtendedEvaluator
from ulib.utils.visualize import display_pert

evaluator = ExtendedEvaluator(pert_model, verbose=True)
print(evaluator.evaluate(dl_eval))
ax = display_pert(pert_model)

In [ ]:
ax = display_pert(pert_model)